# Notebook 19 — Valutazione degli LLM: ROUGE e BLEU in pratica

**Modulo:** 19 — Valutazione degli LLM: ROUGE, BLEU, benchmark
**Tempo stimato:** 1-1,5 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Un'implementazione **da zero** di ROUGE-1, ROUGE-2, ROUGE-L e BLEU,
  verificata a mano sugli stessi esempi numerici della dispensa (Modulo 19,
  § 2-3)
- Un confronto tra la tua implementazione e le librerie standard
  (`rouge-score`, `sacrebleu`)
- Una funzione `valuta_batch` che produce un piccolo report di valutazione
  su più coppie riferimento/generazione
- Due esempi concreti in cui la metrica automatica **diverge** dal giudizio
  umano (Modulo 19, § 1 e § 4), con le tue osservazioni scritte

## Prerequisiti

```bash
pip install rouge-score sacrebleu python-dotenv
```

Non serve nessuna chiave API: lavoriamo solo con testo, in locale, senza
chiamare nessun LLM. Gli esempi di generazione usati in questo notebook
sono scritti a mano, non provengono dal Notebook 18.

---
## Cella 0 — Setup

Questo notebook non ha bisogno di chiavi API. Verifichiamo solo che le
librerie di valutazione siano installate.

In [ ]:
import os
import re
from collections import Counter
from dotenv import load_dotenv

load_dotenv()
print("Nessuna chiave API necessaria per questo notebook: lavoriamo solo su testo locale.")

try:
    from rouge_score import rouge_scorer
    import sacrebleu
    print("OK: rouge-score e sacrebleu installati correttamente.")
except ImportError as e:
    print(f"MANCANTE: installa le librerie con 'pip install rouge-score sacrebleu' ({e})")

---
## Cella 1 — Tokenizzazione semplice

Prima di calcolare qualunque metrica, serve una funzione di tokenizzazione
condivisa. Usiamo una versione minimale: minuscolo + estrazione delle
parole (compresi gli accenti italiani). Non è la stessa tokenizzazione a
sotto-parole usata dai modelli (Modulo 15): qui lavoriamo a livello di
parola intera, come fa ROUGE/BLEU "classico".

In [ ]:
def tokenizza(testo: str) -> list[str]:
    """Tokenizzazione a parole, minuscolo, ignora la punteggiatura."""
    testo = testo.lower()
    return re.findall(r"[a-zàèéìòù]+", testo)


def n_grammi(tokens: list[str], n: int) -> list[tuple]:
    """Restituisce la lista di n-grammi (tuple di n token consecutivi)."""
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


# Prova rapida
esempio_tokens = tokenizza("Il gatto dorme sul divano")
print("Token:   ", esempio_tokens)
print("Bigrammi:", n_grammi(esempio_tokens, 2))

---
## Cella 2 — ROUGE-1 e ROUGE-2 a mano

Implementiamo `rouge_n`, generica per qualunque ordine di n-grammi, seguendo
esattamente le formule del Modulo 19, § 2:

```
Recall    = coincidenze / n-grammi nel riferimento
Precision = coincidenze / n-grammi nella generazione
F1        = 2 × Precision × Recall / (Precision + Recall)
```

Le "coincidenze" si contano come intersezione di **multiset** (ogni
n-gramma ripetuto conta al più tante volte quante compare in entrambi i
lati — è il "clipping" standard di ROUGE/BLEU), non come insieme.

In [ ]:
def rouge_n(riferimento: str, generazione: str, n: int) -> dict:
    """ROUGE-N: recall, precision, f1 basati su n-grammi."""
    tok_rif = tokenizza(riferimento)
    tok_gen = tokenizza(generazione)

    ngrammi_rif = Counter(n_grammi(tok_rif, n))
    ngrammi_gen = Counter(n_grammi(tok_gen, n))

    coincidenze = sum((ngrammi_rif & ngrammi_gen).values())  # intersezione multiset

    totale_rif = sum(ngrammi_rif.values())
    totale_gen = sum(ngrammi_gen.values())

    recall = coincidenze / totale_rif if totale_rif else 0.0
    precision = coincidenze / totale_gen if totale_gen else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {"coincidenze": coincidenze, "recall": round(recall, 4),
            "precision": round(precision, 4), "f1": round(f1, 4)}


# Verifica sull'esempio del Modulo 19, § 2: attesi recall=precision=f1=0.80 (ROUGE-1)
riferimento_1 = "il gatto dorme sul divano"
generazione_1 = "il gatto dorme sul tappeto"

print("ROUGE-1:", rouge_n(riferimento_1, generazione_1, n=1))
print("ROUGE-2:", rouge_n(riferimento_1, generazione_1, n=2))
# Atteso: ROUGE-1 = 0.80 / 0.80 / 0.80 -- ROUGE-2 = 0.75 / 0.75 / 0.75

---
## Cella 3 — ROUGE-L: Longest Common Subsequence

ROUGE-L (Modulo 19, § 2) si basa sulla sottosequenza comune più lunga
(*Longest Common Subsequence*, LCS), non su n-grammi consecutivi:
rispetta l'ordine, ma tollera "buchi" in mezzo. La implementiamo con la
classica programmazione dinamica su matrice.

In [ ]:
def lcs_length(a: list[str], b: list[str]) -> int:
    """Lunghezza della sottosequenza comune più lunga tra due liste di token."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


def rouge_l(riferimento: str, generazione: str) -> dict:
    """ROUGE-L: recall, precision, f1 basati sulla LCS."""
    tok_rif = tokenizza(riferimento)
    tok_gen = tokenizza(generazione)

    lcs = lcs_length(tok_rif, tok_gen)

    recall = lcs / len(tok_rif) if tok_rif else 0.0
    precision = lcs / len(tok_gen) if tok_gen else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {"lcs": lcs, "recall": round(recall, 4),
            "precision": round(precision, 4), "f1": round(f1, 4)}


# Stesso esempio di prima: atteso 0.80 / 0.80 / 0.80 (LCS = 4 su 5 token)
print("ROUGE-L (esempio 1):", rouge_l(riferimento_1, generazione_1))

# Esempio "ordine conta" del Modulo 19, § 2: stesse parole, ordine diverso
riferimento_2 = "il cane corre nel parco"
generazione_2 = "nel parco corre il cane"

print("\nROUGE-1 (esempio 2):", rouge_n(riferimento_2, generazione_2, n=1))
print("ROUGE-L (esempio 2):", rouge_l(riferimento_2, generazione_2))
# Atteso: ROUGE-1 = 1.00 (stesse parole, stesso multiset) ma ROUGE-L = 0.40 (LCS = 2)

**Cosa dimostra questo confronto:** ROUGE-1 non vede alcuna differenza tra
`riferimento_2` e `generazione_2` (stesse cinque parole), mentre ROUGE-L
coglie che la struttura della frase è completamente diversa. È la
"regola pratica" del Modulo 19, § 2: ROUGE-1 alto e ROUGE-L basso segnala
parole giuste ma ordine sbagliato — per questo conviene sempre guardare più
varianti insieme.

---
## Cella 4 — BLEU a mano

Modulo 19, § 3: BLEU combina precision a più ordini di n-grammi con una
media geometrica, penalizzata se la generazione è più corta del
riferimento (*brevity penalty*, BP).

```
BLEU = BP × exp( (1/N) × Σ log(p_n) )
```

In [ ]:
import math

def precisione_ngram(riferimento_tok: list[str], generazione_tok: list[str], n: int) -> float:
    """Precision modificata (clippata) per n-grammi di ordine n."""
    ngrammi_rif = Counter(n_grammi(riferimento_tok, n))
    ngrammi_gen = Counter(n_grammi(generazione_tok, n))
    coincidenze = sum((ngrammi_rif & ngrammi_gen).values())
    totale_gen = sum(ngrammi_gen.values())
    return coincidenze / totale_gen if totale_gen else 0.0


def brevity_penalty(len_riferimento: int, len_generazione: int) -> float:
    """BP = 1 se la generazione è lunga almeno quanto il riferimento, altrimenti penalizza."""
    if len_generazione == 0:
        return 0.0
    if len_generazione >= len_riferimento:
        return 1.0
    return math.exp(1 - len_riferimento / len_generazione)


def bleu(riferimento: str, generazione: str, max_n: int = 2) -> float:
    """BLEU-N semplificato, pesi uniformi su tutti gli ordini fino a max_n."""
    tok_rif = tokenizza(riferimento)
    tok_gen = tokenizza(generazione)

    precisioni = [precisione_ngram(tok_rif, tok_gen, n) for n in range(1, max_n + 1)]
    if any(p == 0 for p in precisioni):
        return 0.0  # convenzione: BLEU si azzera se manca completamente un ordine di n-grammi

    log_medio = sum(math.log(p) for p in precisioni) / max_n
    bp = brevity_penalty(len(tok_rif), len(tok_gen))
    return bp * math.exp(log_medio)


# Verifica sull'esempio del Modulo 19, § 3: atteso BLEU-2 ~= 0.7746
punteggio_bleu = bleu(riferimento_1, generazione_1, max_n=2)
print(f"BLEU-2 (esempio 1): {punteggio_bleu:.4f}")
print(f"Atteso dalla dispensa: ~0.7746")

---
## Cella 5 — Verifica con le librerie standard

La nostra implementazione a mano serve a capire **come funzionano**
davvero ROUGE e BLEU. In pratica, in un progetto reale si usano librerie
testate: `rouge-score` (la stessa libreria usata dal paper originale di
ROUGE) e `sacrebleu` (lo standard de facto per BLEU riproducibile).
Confrontiamo i numeri.

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
punteggi_libreria = scorer.score(riferimento_1, generazione_1)

print("=== rouge-score (libreria) ===")
for chiave, punteggio in punteggi_libreria.items():
    print(f"{chiave}: precision={punteggio.precision:.4f}  recall={punteggio.recall:.4f}  f1={punteggio.fmeasure:.4f}")

print("\n=== Le nostre funzioni ===")
print("rouge_n(n=1):", rouge_n(riferimento_1, generazione_1, n=1))
print("rouge_n(n=2):", rouge_n(riferimento_1, generazione_1, n=2))
print("rouge_l:     ", rouge_l(riferimento_1, generazione_1))

print("\n=== sacrebleu (libreria, scala 0-100) ===")
risultato_sacrebleu = sacrebleu.sentence_bleu(generazione_1, [riferimento_1])
print(f"BLEU (sacrebleu, fino a 4-grammi): {risultato_sacrebleu.score:.2f} / 100  ->  {risultato_sacrebleu.score / 100:.4f}")
print(f"Il nostro BLEU-2:                  {bleu(riferimento_1, generazione_1, max_n=2):.4f}")

**Perché i numeri non coincidono esattamente:** `rouge-score` con
`use_stemmer=False` dovrebbe allinearsi bene alla nostra implementazione su
questo esempio semplice (nessuna forma verbale da ridurre alla radice).
`sacrebleu`, invece, calcola BLEU di default fino a **4-grammi** e applica
una *smoothing function* per evitare punteggi zero quando manca un ordine
di n-grammi: per questo il suo numero non è direttamente confrontabile col
nostro BLEU-2 "didattico". Il principio sottostante — media geometrica di
precision + brevity penalty — è lo stesso in entrambi i casi.

---
## Cella 6 — Le metriche applicate a un batch di generazioni

Un caso più realistico: un piccolo insieme di coppie riferimento/generazione,
come se fossero output di un modello di riassunto (Modulo 19, § 6 — è
esattamente il tipo di tabella che si registra a ogni epoch durante un
fine-tuning, vedi Notebook 18).

In [ ]:
COPPIE_VALUTAZIONE = [
    {
        "riferimento": "Il Comune ha approvato il nuovo piano per la mobilità urbana.",
        "generazione": "Il Comune ha approvato un piano per la mobilità urbana.",
    },
    {
        "riferimento": "La squadra ha vinto la partita grazie a un gol nel finale.",
        "generazione": "La squadra ha vinto grazie a un gol segnato negli ultimi minuti.",
    },
    {
        "riferimento": "Gli scienziati hanno scoperto una nuova specie di rana in Amazzonia.",
        "generazione": "In Amazzonia è stata scoperta una nuova specie di rana dagli scienziati.",
    },
    {
        "riferimento": "Il prezzo del petrolio e salito del tre percento questa settimana.",
        "generazione": "Questa settimana il petrolio e aumentato.",
    },
    {
        "riferimento": "La mostra di arte contemporanea resta aperta fino a dicembre.",
        "generazione": "La mostra chiude in dicembre.",
    },
    {
        "riferimento": "Il nuovo ponte ridurra i tempi di percorrenza tra le due città.",
        "generazione": "Domani forse piovera in tutta la regione.",
    },
]

for coppia in COPPIE_VALUTAZIONE:
    r1 = rouge_n(coppia["riferimento"], coppia["generazione"], n=1)
    rl = rouge_l(coppia["riferimento"], coppia["generazione"])
    b = bleu(coppia["riferimento"], coppia["generazione"], max_n=2)
    print(f"Rif: {coppia['riferimento']}")
    print(f"Gen: {coppia['generazione']}")
    print(f"  ROUGE-1 f1={r1['f1']:.2f}  ROUGE-L f1={rl['f1']:.2f}  BLEU-2={b:.2f}\n")

**Cosa osservare:** l'ultima coppia (ponte / meteo) è completamente
scorrelata — non è un riassunto, è un'altra frase. I punteggi dovrebbero
crollare vicino a zero su tutte le metriche: è un buon controllo di
sanità (*sanity check*) per verificare che le funzioni funzionino come ti
aspetti prima di fidarti dei numeri su casi più ambigui.

---
## Cella 7 — Il limite delle metriche automatiche

Modulo 19, § 1 e § 4: ROUGE e BLEU misurano sovrapposizione lessicale, non
significato. Due casi concreti in cui la metrica automatica diverge
chiaramente dal giudizio umano.

In [ ]:
# Caso 1: riformulazione semanticamente equivalente (Modulo 19, § 1)
riferimento_parafrasi = "Giulia ama il caffè"
generazione_parafrasi = "A Giulia piace davvero bere il caffè"

r1 = rouge_n(riferimento_parafrasi, generazione_parafrasi, n=1)
rl = rouge_l(riferimento_parafrasi, generazione_parafrasi)
b = bleu(riferimento_parafrasi, generazione_parafrasi, max_n=2)

print("=== Caso 1: parafrasi corretta, lessico diverso ===")
print(f"Rif: {riferimento_parafrasi}")
print(f"Gen: {generazione_parafrasi}")
print(f"ROUGE-1 f1={r1['f1']:.2f}  ROUGE-L f1={rl['f1']:.2f}  BLEU-2={b:.2f}")
print("Giudizio umano: corretto, dice esattamente la stessa cosa.")
print("Giudizio della metrica: punteggio basso, sembra un cattivo riassunto.\n")

# Caso 2: negazione -- alta sovrapposizione lessicale, significato opposto
riferimento_negazione = "Il farmaco riduce la pressione sanguigna"
generazione_negazione = "Il farmaco non riduce la pressione sanguigna"

r1b = rouge_n(riferimento_negazione, generazione_negazione, n=1)
rlb = rouge_l(riferimento_negazione, generazione_negazione)
bb = bleu(riferimento_negazione, generazione_negazione, max_n=2)

print("=== Caso 2: una sola parola cambia il significato ===")
print(f"Rif: {riferimento_negazione}")
print(f"Gen: {generazione_negazione}")
print(f"ROUGE-1 f1={r1b['f1']:.2f}  ROUGE-L f1={rlb['f1']:.2f}  BLEU-2={bb:.2f}")
print("Giudizio umano: gravemente sbagliato, dice l'opposto (in un contesto medico, pericoloso).")
print("Giudizio della metrica: punteggio molto alto, quasi tutte le parole coincidono.")

**Le tue osservazioni** (modifica questa cella):

- Nel Caso 1, quale metrica si avvicina di più a un punteggio "giusto"
  secondo te? Nessuna? ...
- Nel Caso 2, cosa succederebbe se usassi solo ROUGE/BLEU per decidere se
  un chatbot medico può andare in produzione? ...
- Che tipo di errore ti sembra più pericoloso in un'applicazione reale: il
  Caso 1 (punteggio ingiustamente basso) o il Caso 2 (punteggio
  ingiustamente alto)? Perché? ...
- Alla luce di questi due casi, perché la dispensa (Modulo 19, § 4) insiste
  sul fatto che la valutazione umana resta indispensabile? ...

---
## Esercizio — completa il codice

**TODO:** scrivi una tua coppia (riferimento, generazione) in cui ti aspetti
che la metrica automatica "sbagli" rispetto al giudizio umano — in una
direzione diversa dai due casi sopra (per esempio: un riassunto molto più
corto ma corretto, oppure un errore di un solo numero/nome proprio che
cambia completamente il senso). Calcola ROUGE-1, ROUGE-L e BLEU-2 e commenta
il risultato.

In [ ]:
# TODO: scrivi qui la tua coppia
mio_riferimento = ""
mia_generazione = ""

# TODO: calcola rouge_n (n=1), rouge_l e bleu sulla tua coppia e stampa i risultati

# TODO: scrivi un commento (come stringa o print) su cosa dimostra il tuo esempio


---
## Sfida finale — un report di valutazione riutilizzabile

Costruisci una funzione `valuta_batch` che:

- accetta una lista di coppie `{"riferimento": ..., "generazione": ...}`
- calcola ROUGE-1, ROUGE-2, ROUGE-L (f1) e BLEU-2 per ciascuna
- calcola anche le **medie** su tutto il batch
- assegna un flag euristico `revisione_umana_consigliata` quando i
  punteggi sono sospetti (molto bassi, **oppure** molto alti con una
  generazione quasi identica al riferimento tranne per un numero ridotto
  di parole — un'euristica grezza per intercettare casi simili al "Caso 2"
  della Cella 7, sapendo che resta comunque imperfetta)

Completa i punti marcati con `# TODO`.

In [ ]:
def valuta_batch(coppie: list[dict], soglia_bassa: float = 0.3, soglia_alta: float = 0.9) -> dict:
    """Calcola ROUGE-1/2/L e BLEU-2 su un batch di coppie riferimento/generazione.

    Restituisce un dict con:
      - risultati: lista di dict per coppia (con tutti i punteggi + flag)
      - medie: dict con la media di ogni metrica sul batch
    """
    risultati = []

    for coppia in coppie:
        rif, gen = coppia["riferimento"], coppia["generazione"]

        # TODO: calcola r1 = rouge_n(rif, gen, n=1), r2 = rouge_n(rif, gen, n=2),
        # rl = rouge_l(rif, gen), b = bleu(rif, gen, max_n=2)

        # TODO: calcola una differenza approssimativa di lunghezza tra rif e gen
        # (in numero di parole tokenizzate) da usare nell'euristica sotto

        # TODO: assegna revisione_umana_consigliata = True se rl["f1"] < soglia_bassa
        # OPPURE se rl["f1"] > soglia_alta e la differenza di lunghezza in parole e' minima
        # (indizio grezzo di un possibile cambio di una sola parola, come la negazione del Caso 2)
        revisione_umana_consigliata = None

        risultati.append({
            "riferimento": rif,
            "generazione": gen,
            # TODO: aggiungi qui rouge1_f1, rouge2_f1, rougeL_f1, bleu2, revisione_umana_consigliata
        })

    # TODO: calcola le medie di rouge1_f1, rouge2_f1, rougeL_f1, bleu2 su tutto risultati
    medie = {}

    return {"risultati": risultati, "medie": medie}


# Applica la funzione al batch della Cella 6 + i due casi limite della Cella 7
batch_completo = COPPIE_VALUTAZIONE + [
    {"riferimento": riferimento_parafrasi, "generazione": generazione_parafrasi},
    {"riferimento": riferimento_negazione, "generazione": generazione_negazione},
]

report = valuta_batch(batch_completo)

for r in report["risultati"]:
    print(r)

print("\nMedie sul batch:", report["medie"])

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 19 — Modulo 19 del Corso Completo di IA Generativa*